# RoPE — Paper-to-Code Mock (Colab)

**Paper:** RoFormer: Enhanced Transformer with Rotary Position Embedding (Su et al., 2021) — https://arxiv.org/abs/2104.09864

Medium mock (~35 min). Fill in the `apply_rope` stub, run the relative-position-invariance demo, then the sanity checks. Reference solution in the last cell.

In [ ]:
import torch
torch.manual_seed(0)

## 1. Implement RoPE (YOUR TASK)

For `x` of shape `(..., seq, dim)` with even `dim`, rotate each consecutive pair `(2i, 2i+1)` by angle `pos * theta_i`, where `theta_i = base^(-2i/dim)`, `base=10000`. Apply it later to q and k before their dot product.

In [ ]:
def apply_rope(x, positions, base=10000.0):
    """x: (..., seq, dim) even dim. positions: (seq,). Returns x rotated."""
    # TODO: half = dim // 2; theta = base ** (-2i/dim) for i in 0..half-1
    # TODO: angles = positions[:, None] * theta[None, :]  -> cos, sin
    # TODO: rotate each pair: even' = even*cos - odd*sin; odd' = even*sin + odd*cos
    # TODO: re-interleave into output of same shape
    raise NotImplementedError("fill me in")

## 2. Demonstrate the property (relative-position invariance)
Fixed content q, k. The attention score q.k must be unchanged when BOTH positions shift by s, because it depends only on the offset m-n. This is the headline property, not a benchmark.

In [ ]:
dim = 8
q = torch.randn(dim)      # fixed CONTENT for the query
k = torch.randn(dim)      # fixed CONTENT for the key
m, n = 5, 2               # absolute positions; offset m - n = 3

def score(content_q, content_k, pm, pn):
    qr = apply_rope(content_q[None, :], torch.tensor([pm]))[0]
    kr = apply_rope(content_k[None, :], torch.tensor([pn]))[0]
    return torch.dot(qr, kr)

base_score = score(q, k, m, n)
print(f"score(q@{m}, k@{n}) = {base_score.item():.6f}   (offset {m-n})")
diffs = []
for s in (1, 3, 7, 50, 123):
    sc = score(q, k, m + s, n + s)
    diffs.append((sc - base_score).abs().item())
    print(f"score(q@{m+s}, k@{n+s}) = {sc.item():.6f}   shift s={s}")
print(f"max abs difference across shifts = {max(diffs):.2e}  (~0 => depends only on m-n)")

## 3. Sanity checks

In [ ]:
# 1) preserves L2 norm (orthogonal rotation)
x = torch.randn(4, dim)
xr = apply_rope(x, torch.arange(4))
assert torch.allclose(x.norm(dim=-1), xr.norm(dim=-1), atol=1e-5)

# 2) position 0 is identity
x0 = torch.randn(1, dim)
assert torch.allclose(apply_rope(x0, torch.tensor([0])), x0, atol=1e-6)

# 3) relative-offset invariance of the score (the core property)
assert torch.allclose(score(q, k, 5, 2), score(q, k, 5 + 17, 2 + 17), atol=1e-4)

# 4) shape preserved
big = torch.randn(2, 6, 10)
assert apply_rope(big, torch.arange(6)).shape == big.shape

# 5) different offsets => different scores (it actually encodes position)
assert not torch.allclose(score(q, k, 5, 2), score(q, k, 5, 0), atol=1e-3)

# 6) composition: rotate(a) then rotate(b) == rotate(a+b)
a, b = 3.0, 4.0
twostep = apply_rope(apply_rope(x0, torch.tensor([a])), torch.tensor([b]))
onestep = apply_rope(x0, torch.tensor([a + b]))
assert torch.allclose(twostep, onestep, atol=1e-5)

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
def apply_rope_solution(x, positions, base=10000.0):
    *_, seq, dim = x.shape
    assert dim % 2 == 0
    half = dim // 2
    i = torch.arange(half, device=x.device, dtype=x.dtype)
    theta = base ** (-2.0 * i / dim)
    angles = positions.to(x.dtype)[:, None] * theta[None, :]
    cos, sin = torch.cos(angles), torch.sin(angles)
    x_even, x_odd = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x_even * cos - x_odd * sin
    out[..., 1::2] = x_even * sin + x_odd * cos
    return out